<a href="https://colab.research.google.com/github/aumer1800/Face_Recognition_Model/blob/main/FaceRecognitionOriginal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Face Recognition Embedding Model — VGGFace2

Clean Google Colab notebook.

Pipeline:
**Kaggle → VGGFace2 → identity split → Custom CNN → classification warm-up → triplet loss → 128-D embeddings → cosine verification → save model**

Start with `TARGET_IDENTITIES = 100` for testing, then increase it.

## 1. Install Kaggle

In [ ]:
!pip install -q kaggle

## 2. Upload `kaggle.json`

Download `kaggle.json` from Kaggle Account/Settings → API and upload it.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving kaggle (2).json to kaggle (2).json


## 3. Configure Kaggle

In [ ]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)

!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

print("Kaggle configured!")

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Kaggle configured!


## 4. Download VGGFace2

In [ ]:
!mkdir -p /content/vggface2_data

!kaggle datasets download -d yakhyokhuja/vggface2-112x112     -p /content/vggface2_data

Dataset URL: https://www.kaggle.com/datasets/yakhyokhuja/vggface2-112x112
License(s): unknown
100% 17.5G/17.5G [16:18<00:00, 19.2MB/s]



## 5. Extract VGGFace2

In [ ]:
import zipfile
import os

DATA_ROOT = "/content/vggface2_data"
RAW_ROOT = "/content/vggface2_data/raw"
ZIP_PATH = "/content/vggface2_data/vggface2-112x112.zip"

os.makedirs(RAW_ROOT, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(RAW_ROOT)

print("Dataset extracted!")

Dataset extracted!


In [ ]:
import os

print("Contents of /content/vggface2_data:")
print(os.listdir("/content/vggface2_data"))

print("\nContents of /content/vggface2_data/raw:")
print(os.listdir("/content/vggface2_data/raw"))

Contents of /content/vggface2_data:
['vggface2-112x112.zip', 'raw']

Contents of /content/vggface2_data/raw:
['vggface2_112x112']


In [ ]:
import os
from collections import Counter

dataset_path = "/content/vggface2_data/raw/vggface2_112x112"

identity_counts = {}

for identity in os.listdir(dataset_path):
    identity_path = os.path.join(dataset_path, identity)

    if os.path.isdir(identity_path):
        image_count = sum(
            1 for f in os.listdir(identity_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )
        identity_counts[identity] = image_count

# Count how many identities have each number of images
image_distribution = Counter(identity_counts.values())

print("Images per identity distribution:\n")

for image_count in sorted(image_distribution):
    num_identities = image_distribution[image_count]
    print(f"{num_identities} identities have {image_count} images")

Images per identity distribution:

1 identities have 87 images
1 identities have 101 images
2 identities have 102 images
2 identities have 104 images
3 identities have 106 images
2 identities have 108 images
1 identities have 110 images
1 identities have 111 images
1 identities have 112 images
2 identities have 113 images
1 identities have 115 images
1 identities have 116 images
3 identities have 117 images
1 identities have 118 images
2 identities have 119 images
1 identities have 120 images
2 identities have 122 images
3 identities have 123 images
2 identities have 124 images
5 identities have 125 images
3 identities have 126 images
3 identities have 127 images
2 identities have 130 images
1 identities have 133 images
2 identities have 134 images
4 identities have 135 images
3 identities have 136 images
2 identities have 137 images
2 identities have 138 images
3 identities have 139 images
3 identities have 140 images
4 identities have 141 images
2 identities have 142 images
1 identit

## 6. Check dataset

In [ ]:
import os

print("Top-level contents:")
for item in os.listdir(RAW_ROOT)[:20]:
    print(item)

Top-level contents:
vggface2_112x112


In [ ]:
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

identity_dirs = []

for root, dirs, files in os.walk(RAW_ROOT):
    image_files = [f for f in files if f.lower().endswith(IMAGE_EXTS)]
    if image_files:
        identity_dirs.append(root)

print("Identity folders found:", len(identity_dirs))

for folder in identity_dirs[:10]:
    print(folder)

Identity folders found: 8631
/content/vggface2_data/raw/vggface2_112x112/id_8137
/content/vggface2_data/raw/vggface2_112x112/id_6516
/content/vggface2_data/raw/vggface2_112x112/id_6828
/content/vggface2_data/raw/vggface2_112x112/id_2483
/content/vggface2_data/raw/vggface2_112x112/id_6943
/content/vggface2_data/raw/vggface2_112x112/id_5719
/content/vggface2_data/raw/vggface2_112x112/id_8354
/content/vggface2_data/raw/vggface2_112x112/id_8223
/content/vggface2_data/raw/vggface2_112x112/id_6728
/content/vggface2_data/raw/vggface2_112x112/id_5282


## 7. Settings

In [ ]:
# ==============================
# SETTINGS
# ==============================

import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

IMG_SIZE = 112
EMBEDDING_DIM = 128

TARGET_IDENTITIES = 8000
MIN_IMAGES_PER_PERSON = 40
IMAGES_PER_PERSON = 40

WARMUP_EPOCHS = 20
TRIPLET_EPOCHS = 40

# Batch-hard triplet sampling
P = 16
K = 4
BATCH_SIZE = P * K

MARGIN = 0.3

print("Settings ready!")
print("Batch size:", BATCH_SIZE)

Settings ready!
Batch size: 64


## 8. Imports and GPU

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 9. Select identities

In [ ]:
# ==============================
# SELECT IDENTITIES
# ==============================

eligible = []

for folder in identity_dirs:

    images = [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(IMAGE_EXTS)
    ]

    if len(images) >= MIN_IMAGES_PER_PERSON:
        eligible.append((folder, images))


print("Eligible identities:", len(eligible))

# Shuffle identities randomly
random.shuffle(eligible)

# Select required number
selected = eligible[:TARGET_IDENTITIES]

print("Selected identities:", len(selected))

# 80% train
# 10% validation
# 10% test

n = len(selected)

train_end = int(n * 0.80)
val_end = int(n * 0.90)

train_people = selected[:train_end]
val_people = selected[train_end:val_end]
test_people = selected[val_end:]

print("Train identities:", len(train_people))
print("Validation identities:", len(val_people))
print("Test identities:", len(test_people))

Eligible identities: 8631
Selected identities: 8000
Train identities: 6400
Validation identities: 800
Test identities: 800


## 11. Create image records

In [ ]:
# ==============================
# CREATE IMAGE RECORDS
# ==============================

def make_records(people):

    records = []

    for label, (folder, images) in enumerate(people):

        images = images.copy()

        # Randomly choose images
        random.shuffle(images)

        # Use only required number
        images = images[:IMAGES_PER_PERSON]

        for image_path in images:
            records.append((image_path, label))

    return records


train_records = make_records(train_people)
val_records = make_records(val_people)
test_records = make_records(test_people)


print("Training images:", len(train_records))
print("Validation images:", len(val_records))
print("Test images:", len(test_records))

Training images: 256000
Validation images: 32000
Test images: 32000


## 12. Transforms

In [ ]:
# ==============================
# IMAGE TRANSFORMS
# ==============================

from torchvision import transforms

train_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(8),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])


eval_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])


print("Transforms ready!")

Transforms ready!


## 13. Dataset and DataLoaders

In [ ]:
# ==============================
# DATASETS
# ==============================

from PIL import Image
from torch.utils.data import Dataset, DataLoader


class FaceDataset(Dataset):

    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):

        path, label = self.records[index]

        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


train_dataset = FaceDataset(
    train_records,
    train_transform
)

val_dataset = FaceDataset(
    val_records,
    eval_transform
)

test_dataset = FaceDataset(
    test_records,
    eval_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("DataLoaders ready!")

DataLoaders ready!


## 14. Custom CNN embedding model

In [ ]:
# ==============================
# CUSTOM CNN EMBEDDING MODEL
# ==============================

import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True),

            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.block(x)


class FaceEmbeddingModel(nn.Module):

    def __init__(
        self,
        num_classes,
        embedding_dim=128
    ):

        super().__init__()

        self.features = nn.Sequential(

            ConvBlock(3, 32),

            ConvBlock(32, 64),

            ConvBlock(64, 128),

            ConvBlock(128, 256),

            ConvBlock(256, 512)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.embedding = nn.Sequential(

            nn.Linear(512, 256),

            nn.BatchNorm1d(256),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(256, embedding_dim)
        )

        # Temporary classifier
        # Only used during warm-up
        self.classifier = nn.Linear(
            embedding_dim,
            num_classes
        )


    def forward(
        self,
        x,
        return_embedding=False
    ):

        x = self.features(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        embedding = self.embedding(x)

        # Used during triplet training
        # and final face recognition
        if return_embedding:

            return F.normalize(
                embedding,
                p=2,
                dim=1
            )

        # Used only for classification warm-up
        return self.classifier(embedding)

In [ ]:
# ==============================
# CREATE FRESH MODEL
# ==============================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

num_classes = len(train_people)

model = FaceEmbeddingModel(
    num_classes=num_classes,
    embedding_dim=EMBEDDING_DIM
).to(device)


print("Device:", device)
print("Number of classes:", num_classes)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Total parameters: {total_params:,}"
)

Device: cuda
Number of classes: 6400
Total parameters: 5,704,544


## 15. Classification warm-up

In [ ]:
# ============================================================
# CLASSIFICATION WARM-UP — CHECKPOINT EVERY 5 EPOCHS
# ============================================================

import os
import torch
import torch.nn as nn

from google.colab import drive

# ------------------------------------------------------------
# MOUNT GOOGLE DRIVE
# ------------------------------------------------------------

drive.mount("/content/drive")


# ------------------------------------------------------------
# CHECKPOINT DIRECTORY
# ------------------------------------------------------------

CHECKPOINT_DIR = "/content/drive/MyDrive/face_recognition_checkpoints"

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

print("Checkpoint directory:")
print(CHECKPOINT_DIR)


# ------------------------------------------------------------
# LOSS
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)


# ------------------------------------------------------------
# OPTIMIZER
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# SCHEDULER
# ------------------------------------------------------------

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=WARMUP_EPOCHS
)


# ------------------------------------------------------------
# RESUME SETTINGS
# ------------------------------------------------------------

START_EPOCH = 0

# Change this to True when you want to resume
RESUME_TRAINING = True


# ------------------------------------------------------------
# FIND LATEST CHECKPOINT
# ------------------------------------------------------------

latest_checkpoint = None

if os.path.exists(CHECKPOINT_DIR):

    checkpoints = [
        f for f in os.listdir(CHECKPOINT_DIR)
        if f.startswith("warmup_epoch_")
        and f.endswith(".pth")
    ]

    if checkpoints:

        checkpoints.sort(
            key=lambda x: int(
                x.replace("warmup_epoch_", "")
                 .replace(".pth", "")
            )
        )

        latest_checkpoint = os.path.join(
            CHECKPOINT_DIR,
            checkpoints[-1]
        )


# ------------------------------------------------------------
# LOAD LATEST CHECKPOINT
# ------------------------------------------------------------

if RESUME_TRAINING and latest_checkpoint is not None:

    print("=" * 70)
    print("RESUMING WARM-UP TRAINING")
    print("=" * 70)

    checkpoint = torch.load(
        latest_checkpoint,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    START_EPOCH = checkpoint["epoch"]

    print(
        f"Loaded checkpoint: "
        f"{os.path.basename(latest_checkpoint)}"
    )

    print(
        f"Resuming from epoch: "
        f"{START_EPOCH + 1}"
    )

    print(
        f"Previous loss: "
        f"{checkpoint['loss']:.4f}"
    )

    print(
        f"Previous accuracy: "
        f"{checkpoint['accuracy']:.2f}%"
    )

    print("=" * 70)

else:

    print("=" * 70)
    print("STARTING WARM-UP TRAINING FROM EPOCH 1")
    print("=" * 70)


# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

for epoch in range(
    START_EPOCH,
    WARMUP_EPOCHS
):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # CLASSIFICATION OUTPUT
        # ----------------------------------------------------

        logits = model(images)

        loss = criterion(
            logits,
            labels
        )

        # ----------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------

        loss.backward()

        # Prevent very large gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        total_loss += loss.item()

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    # --------------------------------------------------------
    # SCHEDULER
    # --------------------------------------------------------

    scheduler.step()

    accuracy = 100 * correct / total

    average_loss = (
        total_loss / len(train_loader)
    )

    current_lr = optimizer.param_groups[0]["lr"]

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"Epoch {epoch + 1:02d}/{WARMUP_EPOCHS} | "
        f"Loss: {average_loss:.4f} | "
        f"Accuracy: {accuracy:.2f}% | "
        f"LR: {current_lr:.7f}"
    )

    # --------------------------------------------------------
    # SAVE EVERY 5 EPOCHS
    # --------------------------------------------------------

    completed_epoch = epoch + 1

    if (
        completed_epoch % 5 == 0
        or completed_epoch == WARMUP_EPOCHS
    ):

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"warmup_epoch_{completed_epoch}.pth"
        )

        torch.save(
            {
                "epoch": completed_epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "loss":
                    average_loss,

                "accuracy":
                    accuracy,

                "learning_rate":
                    current_lr,

                "warmup_epochs":
                    WARMUP_EPOCHS
            },
            checkpoint_path
        )

        print()
        print("✓ CHECKPOINT SAVED")
        print(
            f"  Epoch: {completed_epoch}"
        )
        print(
            f"  File: {checkpoint_path}"
        )
        print()

Mounted at /content/drive
Checkpoint directory:
/content/drive/MyDrive/face_recognition_checkpoints
STARTING WARM-UP TRAINING FROM EPOCH 1
Epoch 01/20 | Loss: 8.1957 | Accuracy: 0.36% | LR: 0.0009938
Epoch 02/20 | Loss: 6.4644 | Accuracy: 5.83% | LR: 0.0009755
Epoch 03/20 | Loss: 5.3765 | Accuracy: 18.41% | LR: 0.0009455
Epoch 04/20 | Loss: 4.6186 | Accuracy: 32.69% | LR: 0.0009045
Epoch 05/20 | Loss: 4.0835 | Accuracy: 44.28% | LR: 0.0008536

✓ CHECKPOINT SAVED
  Epoch: 5
  File: /content/drive/MyDrive/face_recognition_checkpoints/warmup_epoch_5.pth

Epoch 06/20 | Loss: 3.7071 | Accuracy: 52.84% | LR: 0.0007939
Epoch 07/20 | Loss: 3.4352 | Accuracy: 59.24% | LR: 0.0007270
Epoch 08/20 | Loss: 3.2202 | Accuracy: 64.13% | LR: 0.0006545
Epoch 09/20 | Loss: 3.0503 | Accuracy: 68.20% | LR: 0.0005782
Epoch 10/20 | Loss: 2.9025 | Accuracy: 71.79% | LR: 0.0005000

✓ CHECKPOINT SAVED
  Epoch: 10
  File: /content/drive/MyDrive/face_recognition_checkpoints/warmup_epoch_10.pth

Epoch 11/20 | Loss:

## 16. Identity-balanced sampler for triplet learning

In [ ]:
# ==============================
# P-K SAMPLER
# ==============================

from torch.utils.data import Sampler
from collections import defaultdict


class PKSampler(Sampler):

    def __init__(
        self,
        records,
        P=16,
        K=4
    ):

        self.records = records
        self.P = P
        self.K = K

        self.by_label = defaultdict(list)

        for index, (_, label) in enumerate(records):

            self.by_label[label].append(index)

        self.labels = list(
            self.by_label.keys()
        )

    def __iter__(self):

        labels = self.labels.copy()

        random.shuffle(labels)

        for start in range(
            0,
            len(labels) - self.P + 1,
            self.P
        ):

            chosen_labels = labels[
                start:start + self.P
            ]

            batch = []

            for label in chosen_labels:

                indices = self.by_label[label]

                if len(indices) >= self.K:

                    chosen = random.sample(
                        indices,
                        self.K
                    )

                else:

                    chosen = random.choices(
                        indices,
                        k=self.K
                    )

                batch.extend(chosen)

            yield from batch

    def __len__(self):

        batches = len(self.labels) // self.P

        return (
            batches
            * self.P
            * self.K
        )


# ==============================
# CREATE TRIPLET DATALOADER
# ==============================

triplet_sampler = PKSampler(
    train_records,
    P=P,
    K=K
)

triplet_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=triplet_sampler,
    num_workers=2,
    pin_memory=True
)

print("Triplet DataLoader ready!")
print("Batch size:", BATCH_SIZE)
print("Identities per batch (P):", P)
print("Images per identity (K):", K)

Triplet DataLoader ready!
Batch size: 64
Identities per batch (P): 16
Images per identity (K): 4


## 17. Batch-hard triplet loss

In [ ]:
# ==============================
# BATCH-HARD TRIPLET LOSS
# ==============================

def batch_hard_triplet_loss(
    embeddings,
    labels,
    margin=0.3
):

    # Pairwise distances
    distances = torch.cdist(
        embeddings,
        embeddings,
        p=2
    )

    labels = labels.unsqueeze(1)

    # Same identity
    positive_mask = labels.eq(
        labels.T
    )

    # Different identity
    negative_mask = ~positive_mask

    # Remove self-comparisons
    eye = torch.eye(
        len(embeddings),
        device=embeddings.device,
        dtype=torch.bool
    )

    positive_mask = (
        positive_mask & ~eye
    )

    # Hardest positive
    hardest_positive = distances.masked_fill(
        ~positive_mask,
        -1
    ).max(dim=1).values

    # Hardest negative
    hardest_negative = distances.masked_fill(
        ~negative_mask,
        float("inf")
    ).min(dim=1).values

    # Triplet loss
    loss = F.relu(
        hardest_positive
        - hardest_negative
        + margin
    )

    return loss.mean()

## 18. Train the embedding with triplet loss

In [ ]:
# ============================================================
# ACTUAL TRIPLET TRAINING
# CHECKPOINT SAVED AFTER EVERY EPOCH
# ============================================================

import os
import torch


# ============================================================
# GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")


# ============================================================
# CHECKPOINT DIRECTORY
# ============================================================

TRIPLET_CHECKPOINT_DIR = (
    "/content/drive/MyDrive/face_recognition_checkpoints/triplet"
)

os.makedirs(
    TRIPLET_CHECKPOINT_DIR,
    exist_ok=True
)

print("=" * 70)
print("TRIPLET CHECKPOINT DIRECTORY")
print("=" * 70)
print(TRIPLET_CHECKPOINT_DIR)


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    list(model.features.parameters())
    +
    list(model.embedding.parameters()),
    lr=3e-4,
    weight_decay=1e-4
)


# ============================================================
# SCHEDULER
# ============================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TRIPLET_EPOCHS
)


# ============================================================
# RESUME SETTINGS
# ============================================================

RESUME_TRIPLET_TRAINING = True

START_EPOCH = 0


# ============================================================
# FIND LATEST CHECKPOINT
# ============================================================

latest_checkpoint = None

if os.path.exists(TRIPLET_CHECKPOINT_DIR):

    checkpoint_files = [
        f
        for f in os.listdir(TRIPLET_CHECKPOINT_DIR)
        if f.startswith("triplet_epoch_")
        and f.endswith(".pth")
    ]

    if checkpoint_files:

        checkpoint_files.sort(
            key=lambda x: int(
                x.replace("triplet_epoch_", "")
                 .replace(".pth", "")
            )
        )

        latest_checkpoint = os.path.join(
            TRIPLET_CHECKPOINT_DIR,
            checkpoint_files[-1]
        )


# ============================================================
# LOAD LATEST CHECKPOINT
# ============================================================

if (
    RESUME_TRIPLET_TRAINING
    and latest_checkpoint is not None
):

    print()
    print("=" * 70)
    print("RESUMING TRIPLET TRAINING")
    print("=" * 70)

    checkpoint = torch.load(
        latest_checkpoint,
        map_location=device
    )

    # --------------------------------------------------------
    # LOAD MODEL
    # --------------------------------------------------------

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    # --------------------------------------------------------
    # LOAD OPTIMIZER
    # --------------------------------------------------------

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    # --------------------------------------------------------
    # LOAD SCHEDULER
    # --------------------------------------------------------

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    # --------------------------------------------------------
    # RESUME EPOCH
    # --------------------------------------------------------

    START_EPOCH = checkpoint["epoch"]

    print(
        "Loaded checkpoint:"
    )

    print(
        os.path.basename(
            latest_checkpoint
        )
    )

    print()

    print(
        f"Completed epoch : {START_EPOCH}"
    )

    print(
        f"Previous loss   : "
        f"{checkpoint['loss']:.4f}"
    )

    print(
        f"Previous LR     : "
        f"{checkpoint['learning_rate']:.6f}"
    )

    print("=" * 70)
    print()

else:

    print()
    print("=" * 70)
    print("STARTING TRIPLET TRAINING FROM EPOCH 1")
    print("=" * 70)
    print()


# ============================================================
# ACTUAL TRIPLET TRAINING
# ============================================================

for epoch in range(
    START_EPOCH,
    TRIPLET_EPOCHS
):

    model.train()

    total_loss = 0.0

    # --------------------------------------------------------
    # TRAINING BATCHES
    # --------------------------------------------------------

    for images, labels in triplet_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # CLEAR GRADIENTS
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # GET 128-D FACE EMBEDDINGS
        # ----------------------------------------------------

        embeddings = model(
            images,
            return_embedding=True
        )

        # ----------------------------------------------------
        # BATCH-HARD TRIPLET LOSS
        # ----------------------------------------------------

        loss = batch_hard_triplet_loss(
            embeddings,
            labels,
            margin=MARGIN
        )

        # ----------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------

        loss.backward()

        # ----------------------------------------------------
        # GRADIENT CLIPPING
        # ----------------------------------------------------

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        # ----------------------------------------------------
        # UPDATE MODEL
        # ----------------------------------------------------

        optimizer.step()

        # ----------------------------------------------------
        # ACCUMULATE LOSS
        # ----------------------------------------------------

        total_loss += loss.item()


    # ========================================================
    # SCHEDULER
    # ========================================================

    scheduler.step()


    # ========================================================
    # CALCULATE AVERAGE LOSS
    # ========================================================

    average_loss = (
        total_loss /
        len(triplet_loader)
    )


    # ========================================================
    # CURRENT LEARNING RATE
    # ========================================================

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )


    # ========================================================
    # COMPLETED EPOCH
    # ========================================================

    completed_epoch = epoch + 1


    # ========================================================
    # PRINT PROGRESS
    # ========================================================

    print("=" * 70)

    print(
        f"Triplet Epoch "
        f"{completed_epoch:02d}/"
        f"{TRIPLET_EPOCHS}"
    )

    print(
        f"Loss : {average_loss:.4f}"
    )

    print(
        f"LR   : {current_lr:.6f}"
    )


    # ========================================================
    # SAVE CHECKPOINT AFTER EVERY EPOCH
    # ========================================================

    checkpoint_path = os.path.join(
        TRIPLET_CHECKPOINT_DIR,
        f"triplet_epoch_{completed_epoch}.pth"
    )

    torch.save(
        {
            # Current epoch
            "epoch": completed_epoch,

            # Model
            "model_state_dict":
                model.state_dict(),

            # Optimizer
            "optimizer_state_dict":
                optimizer.state_dict(),

            # Scheduler
            "scheduler_state_dict":
                scheduler.state_dict(),

            # Training information
            "loss":
                average_loss,

            "learning_rate":
                current_lr,

            # Configuration
            "triplet_epochs":
                TRIPLET_EPOCHS,

            "margin":
                MARGIN
        },
        checkpoint_path
    )


    # ========================================================
    # CONFIRM SAVE
    # ========================================================

    print(
        f"✓ CHECKPOINT SAVED"
    )

    print(
        f"  {checkpoint_path}"
    )

    print("=" * 70)
    print()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TRIPLET CHECKPOINT DIRECTORY
/content/drive/MyDrive/face_recognition_checkpoints/triplet

STARTING TRIPLET TRAINING FROM EPOCH 1

Triplet Epoch 01/40
Loss : 0.2639
LR   : 0.000300
✓ CHECKPOINT SAVED
  /content/drive/MyDrive/face_recognition_checkpoints/triplet/triplet_epoch_1.pth

Triplet Epoch 02/40
Loss : 0.2537
LR   : 0.000298
✓ CHECKPOINT SAVED
  /content/drive/MyDrive/face_recognition_checkpoints/triplet/triplet_epoch_2.pth

Triplet Epoch 03/40
Loss : 0.2503
LR   : 0.000296
✓ CHECKPOINT SAVED
  /content/drive/MyDrive/face_recognition_checkpoints/triplet/triplet_epoch_3.pth

Triplet Epoch 04/40
Loss : 0.2494
LR   : 0.000293
✓ CHECKPOINT SAVED
  /content/drive/MyDrive/face_recognition_checkpoints/triplet/triplet_epoch_4.pth

Triplet Epoch 05/40
Loss : 0.2514
LR   : 0.000289
✓ CHECKPOINT SAVED
  /content/drive/MyDrive/face_recognition_checkpoints/triplet/tr

## 19. Extract 128-D face embeddings

In [ ]:
# ==============================
# EXTRACT FACE EMBEDDINGS
# ==============================

@torch.no_grad()
def get_embeddings(
    model,
    loader
):

    model.eval()

    embeddings = []
    labels = []

    for images, batch_labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        batch_embeddings = model(
            images,
            return_embedding=True
        )

        embeddings.append(
            batch_embeddings.cpu()
        )

        labels.append(
            batch_labels
        )

    embeddings = torch.cat(
        embeddings
    )

    labels = torch.cat(
        labels
    )

    return embeddings, labels


# ==============================
# VALIDATION EMBEDDINGS
# ==============================

val_embeddings, val_labels = get_embeddings(
    model,
    val_loader
)


# ==============================
# TEST EMBEDDINGS
# ==============================

test_embeddings, test_labels = get_embeddings(
    model,
    test_loader
)


print(
    "Validation embedding shape:",
    val_embeddings.shape
)

print(
    "Test embedding shape:",
    test_embeddings.shape
)


# ==============================
# EMBEDDING HEALTH CHECK
# ==============================

print(
    "Embedding mean:",
    test_embeddings.mean().item()
)

print(
    "Embedding std:",
    test_embeddings.std().item()
)

print(
    "Average embedding norm:",
    test_embeddings.norm(
        dim=1
    ).mean().item()
)

Validation embedding shape: torch.Size([32000, 128])
Test embedding shape: torch.Size([32000, 128])
Embedding mean: -2.331300129299052e-05
Embedding std: 0.0883883535861969
Average embedding norm: 1.0


## 20. Cosine verification

In [ ]:
# ============================================================
# COSINE VERIFICATION PAIRS
# ============================================================

import random
import numpy as np
import torch
import torch.nn.functional as F


def create_verification_pairs(
    embeddings,
    labels,
    max_genuine=2000,
    max_impostor=2000
):

    embeddings = F.normalize(
        embeddings,
        p=2,
        dim=1
    )

    embeddings_np = embeddings.cpu().numpy()
    labels_np = labels.cpu().numpy()

    identity_to_indices = {}

    for index, label in enumerate(labels_np):

        label = int(label)

        if label not in identity_to_indices:
            identity_to_indices[label] = []

        identity_to_indices[label].append(index)


    genuine_pairs = []
    impostor_pairs = []


    # ========================================================
    # GENUINE PAIRS
    # ========================================================

    identities = list(
        identity_to_indices.keys()
    )

    random.shuffle(identities)

    for identity in identities:

        indices = identity_to_indices[
            identity
        ]

        if len(indices) < 2:
            continue

        pairs = []

        for i in range(len(indices)):

            for j in range(
                i + 1,
                len(indices)
            ):

                pairs.append(
                    (
                        indices[i],
                        indices[j]
                    )
                )

        random.shuffle(pairs)

        for i, j in pairs:

            score = np.dot(
                embeddings_np[i],
                embeddings_np[j]
            )

            genuine_pairs.append(
                score
            )

            if len(genuine_pairs) >= max_genuine:
                break

        if len(genuine_pairs) >= max_genuine:
            break


    # ========================================================
    # IMPOSTOR PAIRS
    # ========================================================

    attempts = 0
    max_attempts = max_impostor * 20

    while (
        len(impostor_pairs) < max_impostor
        and attempts < max_attempts
    ):

        identity1, identity2 = random.sample(
            identities,
            2
        )

        index1 = random.choice(
            identity_to_indices[identity1]
        )

        index2 = random.choice(
            identity_to_indices[identity2]
        )

        score = np.dot(
            embeddings_np[index1],
            embeddings_np[index2]
        )

        impostor_pairs.append(
            score
        )

        attempts += 1


    return (
        np.asarray(genuine_pairs),
        np.asarray(impostor_pairs)
    )


print("Verification pair function ready!")

Verification pair function ready!


## 21. Find verification threshold

In [ ]:
# ============================================================
# VALIDATION THRESHOLD
# ============================================================

val_genuine, val_impostor = create_verification_pairs(
    val_embeddings,
    val_labels,
    max_genuine=2000,
    max_impostor=2000
)


val_scores = np.concatenate([
    val_genuine,
    val_impostor
])


val_targets = np.concatenate([
    np.ones(len(val_genuine)),
    np.zeros(len(val_impostor))
])


best_threshold = 0.0
best_accuracy = 0.0


for threshold in np.arange(
    0.10,
    1.00,
    0.005
):

    predictions = (
        val_scores >= threshold
    ).astype(int)

    accuracy = np.mean(
        predictions == val_targets
    )

    if accuracy > best_accuracy:

        best_accuracy = accuracy
        best_threshold = threshold


print("=" * 60)
print("VALIDATION THRESHOLD")
print("=" * 60)

print(
    f"Best threshold : "
    f"{best_threshold:.3f}"
)

print(
    f"Validation accuracy : "
    f"{best_accuracy * 100:.2f}%"
)

print(
    "Genuine pairs :",
    len(val_genuine)
)

print(
    "Impostor pairs:",
    len(val_impostor)
)

print("=" * 60)

VALIDATION THRESHOLD
Best threshold : 0.305
Validation accuracy : 96.38%
Genuine pairs : 2000
Impostor pairs: 2000


In [ ]:
# ============================================================
# FINAL TEST VERIFICATION
# ============================================================

test_genuine, test_impostor = create_verification_pairs(
    test_embeddings,
    test_labels,
    max_genuine=2000,
    max_impostor=2000
)


test_scores = np.concatenate([
    test_genuine,
    test_impostor
])


test_targets = np.concatenate([
    np.ones(len(test_genuine)),
    np.zeros(len(test_impostor))
])


# ============================================================
# USE VALIDATION THRESHOLD
# ============================================================

test_predictions = (
    test_scores >= best_threshold
).astype(int)


# ============================================================
# METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


test_accuracy = accuracy_score(
    test_targets,
    test_predictions
)

test_precision = precision_score(
    test_targets,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    test_targets,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    test_targets,
    test_predictions,
    zero_division=0
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

tn, fp, fn, tp = confusion_matrix(
    test_targets,
    test_predictions,
    labels=[0, 1]
).ravel()


# ============================================================
# FAR / FRR
# ============================================================

far = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0
)

frr = (
    fn / (fn + tp)
    if (fn + tp) > 0
    else 0
)


# ============================================================
# RESULTS
# ============================================================

print("=" * 70)
print("FACE VERIFICATION — FINAL TEST RESULTS")
print("=" * 70)

print(
    f"Threshold  : {best_threshold:.3f}"
)

print(
    f"Accuracy   : {test_accuracy * 100:.2f}%"
)

print(
    f"Precision  : {test_precision * 100:.2f}%"
)

print(
    f"Recall     : {test_recall * 100:.2f}%"
)

print(
    f"F1 Score   : {test_f1 * 100:.2f}%"
)

print("-" * 70)

print(
    f"True Negatives  : {tn}"
)

print(
    f"False Positives : {fp}"
)

print(
    f"False Negatives : {fn}"
)

print(
    f"True Positives  : {tp}"
)

print("-" * 70)

print(
    f"FAR : {far * 100:.2f}%"
)

print(
    f"FRR : {frr * 100:.2f}%"
)

print("-" * 70)

print(
    f"Mean Genuine Similarity  : "
    f"{test_genuine.mean():.4f}"
)

print(
    f"Mean Impostor Similarity : "
    f"{test_impostor.mean():.4f}"
)

print("=" * 70)

FACE VERIFICATION — FINAL TEST RESULTS
Threshold  : 0.305
Accuracy   : 93.65%
Precision  : 93.35%
Recall     : 94.00%
F1 Score   : 93.67%
----------------------------------------------------------------------
True Negatives  : 1866
False Positives : 134
False Negatives : 120
True Positives  : 1880
----------------------------------------------------------------------
FAR : 6.70%
FRR : 6.00%
----------------------------------------------------------------------
Mean Genuine Similarity  : 0.6531
Mean Impostor Similarity : 0.0061


## 22. Save the embedding model

In [ ]:
# ============================================================
# SAVE FINAL EMBEDDING MODEL PERMANENTLY TO GOOGLE DRIVE
# ============================================================

import os
import torch

from google.colab import drive

# ------------------------------------------------------------
# MOUNT GOOGLE DRIVE
# ------------------------------------------------------------

drive.mount("/content/drive")


# ------------------------------------------------------------
# CREATE MODEL DIRECTORY
# ------------------------------------------------------------

MODEL_DIR = (
    "/content/drive/MyDrive/"
    "face_recognition_models"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# FINAL MODEL PATH
# ------------------------------------------------------------

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "face_embedding_model.pth"
)


# ------------------------------------------------------------
# SAVE MODEL
# ------------------------------------------------------------

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "embedding_dim":
            EMBEDDING_DIM,

        "img_size":
            IMG_SIZE,

        "threshold":
            float(best_threshold),

        "test_accuracy":
            float(test_accuracy)
    },
    MODEL_PATH
)


# ------------------------------------------------------------
# VERIFY FILE
# ------------------------------------------------------------

if os.path.exists(MODEL_PATH):

    file_size_mb = (
        os.path.getsize(MODEL_PATH)
        / (1024 * 1024)
    )

    print("=" * 60)
    print("FINAL MODEL SAVED SUCCESSFULLY")
    print("=" * 60)

    print(
        "Path:",
        MODEL_PATH
    )

    print(
        "File size:",
        f"{file_size_mb:.2f} MB"
    )

    print(
        "Embedding dimension:",
        EMBEDDING_DIM
    )

    print(
        "Threshold:",
        best_threshold
    )

    print(
        "Test accuracy:",
        f"{test_accuracy * 100:.2f}%"
    )

    print("=" * 60)

else:

    print("ERROR: Model was not saved!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FINAL MODEL SAVED SUCCESSFULLY
Path: /content/drive/MyDrive/face_recognition_models/face_embedding_model.pth
File size: 21.80 MB
Embedding dimension: 128
Threshold: 0.30500000000000016
Test accuracy: 93.65%


## 23. Download the model

In [ ]:
from google.colab import files

files.download(MODEL_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Final deployment idea

For your attendance system:

**Camera → face detection/alignment → 112×112 face → CNN → 128-D embedding → cosine similarity → employee identity**

The classifier used during warm-up is not the deployment identity database. The important output for recognition is the **128-D embedding**.